# SDPA 精度验收


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 验收基准

教学版对照 `torch.nn.functional.scaled_dot_product_attention`（下称 torch SDPA）：

```python
import torch
import torch.nn.functional as F

golden = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=True)  # 无 mask 时用 False，不传 None
```

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">项</th><th style="text-align:left;">方案</th></tr>
<tr><td>输入构造</td><td>torch.manual_seed 固定；普通随机输入并补零 Q/K、常量与大幅值输入；退化情形也必须测试</td></tr>
<tr><td>比较方式</td><td>逐输出 <code>torch.testing.assert_close(out, golden, rtol=1e-3, atol=1e-3)</code>（本课程 RFC 验收阈值，不是所有 FP16 算子的通用标准）</td></tr>
<tr><td>极端用例</td><td>极大 logits（exp 溢出防护——正是 online softmax 减最大值的作用）单独构造验证；causal/无 mask 两档各自回归（无 mask 下不存在全遮蔽行）</td></tr>
<tr><td>多形状回归</td><td>S ∈ {1K, 2K, 4K, 8K}、H ∈ {64, 128}、causal/无 mask 组合</td></tr>
</table>


## 2. 数值要点

1. **中间量 FP32**：s/acc/l/m 全程 FP32（L0C 累加天然 FP32；p 块转 FP16 存 UB 时记录精度影响）；
2. **m 减法防溢出**：exp 前减去行最大值是数值稳定的关键——验收时构造大 logits 用例（如 s 值域 ±100）验证不溢出；
3. **l=0 防护**：本章等长标准 causal/无 mask 不产生全遮蔽行；第 11 章组合场景若 l=0，约定输出 0，需在 online 更新时就避免 NaN，并使用安全分母与分支选择。不能给所有行无条件加 eps 改变公式；
4. **差异定位**：不匹配时输出首个错误位置（行号），结合该行所属 Q 块与 K 块边界反推（第 8 章最小失败用例法）。


## 3. 验收入口

10.04 已提供 `src/sdpa_reference.py` 中的 `run_sdpa`。以下函数通过 run_sdpa 参数接收该包装；从章节目录执行 `python src/sdpa_reference.py` 可跑默认输入、零/常量/大 logits 和尾块用例，加 `--full` 扩为 RFC 全形状、B=1/2、两个 seed 和两种 mask。首次编译耗时不计入稳定执行性能。


In [ ]:
import torch.nn.functional as F

def validate(run_sdpa, S, H, causal, B=1, seed=0, device=None):
    if device is None:
        device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
    torch.manual_seed(seed)
    q, k, v = (torch.randn(B, S, H, dtype=torch.float16, device=device) for _ in range(3))
    out = run_sdpa(q, k, v, causal=causal)  # causal 必须实际传到被测实现
    torch.npu.synchronize()
    golden = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0, is_causal=causal)
    assert out.shape == q.shape and out.dtype == q.dtype
    assert torch.isfinite(out).all()
    torch.testing.assert_close(out, golden, rtol=1e-3, atol=1e-3)
    print(f"SDPA B={B} S={S} H={H} causal={causal} passed")

回归覆盖 B=1 及 B>1、S 四档、H 两档、两种 mask 和多个 seed；另用零/常量/大幅值数据测试数学边界。记录 torch/torch_npu 版本、SDPA 后端与误差统计。


### 用结构化输入区分错误来源

Q=K=0 时，无 mask 输出是全部 V 的均值；causal 的第 r 行是 V[:r+1] 的均值。这个用例不依赖复杂 logits，仍能发现 mask 偏一位和归一化分母错误。再令 V 恒等于 0.125：每个非空行都应返回 0.125，若误差只从第二个 KV 块开始，优先核对旧 acc/l 是否使用同一个 alpha。

误差记录必须带 B/S/H、mask、seed、输入类型、首个坐标与最大绝对误差。随机输入通过后还要测大 logits；若 FP16 p 的舍入导致误差超过阈值，应改实现或输入格式，不能自行放宽 RFC 验收值。


## 4. 课后练习

1. （实践）补全极端用例：构造 logits 值域很大的输入（如通过大 α 放缩使 QK^T 元素达 ±100），验证 online softmax 的 m 减法防溢出——与 torch SDPA 输出契约一致吗？
2. （计算）为什么基线尝试把 p 块转为 FP16，却把 acc 保留为 FP32？p 转 FP16 后还需要验证什么？（从"和的规模与舍入误差累积"角度回答）

In [ ]:
!cat ./answer/10.05_answer.txt

## 5. 本节小结

- 基准 torch SDPA + 逐输出 all-close + 多形状回归；
- 数值三要点：**FP32 中间量、m 减法防溢出、l=0 防护**；
- 失败定位回到最小失败用例法——行号 → Q/K 块边界。